In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
wine = load_wine()
#print(list(wine.keys()))
print(wine.feature_names)
df = pd.DataFrame(wine.data, columns=wine.feature_names)

#develop linear regression: Goal predict Alcohol content using the remaining 12 features , which 2 features influence alcohol the most

target="alcohol"

features = df.columns.drop(target)
X = df.drop(columns=["alcohol", "target"], errors="ignore")

#X = df[features]
y = df[target]
#print("X: ", list(X)) #print("Y:", list(y))
#Lin regre
from sklearn.linear_model import LinearRegression

#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model = LinearRegression()
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)

#from sklearn.linear_model import LogisticRegression


['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


In [ ]:
y = df["alcohol"]
X = df.drop(columns=["alcohol", "target"], errors="ignore")
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


model = LinearRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

#print(y_pred)

# Rank features by absolute standardized coefficient
importance = pd.DataFrame({
    "feature": features,
    "std_coef": model.coef_,
    "abs_influence": np.abs(model.coef_),
}).sort_values(by="abs_influence", ascending=False)

print("\nTop features by standardized influence:")
print(importance.to_string(index=False))

print("\nTop 2 features:")
print(importance.head(2))

Test R²:   0.6889
Test RMSE: 0.4309
Test MAE:  0.3675

Top features by standardized influence:
                     feature  std_coef  abs_influence
             color_intensity  0.394276       0.394276
                     proline  0.298278       0.298278
           alcalinity_of_ash -0.161299       0.161299
                  malic_acid  0.128679       0.128679
                         hue  0.094172       0.094172
od280/od315_of_diluted_wines  0.085861       0.085861
             proanthocyanins -0.083039       0.083039
                         ash  0.074655       0.074655
               total_phenols  0.058450       0.058450
                   magnesium -0.026295       0.026295
        nonflavanoid_phenols -0.015891       0.015891
                  flavanoids -0.010809       0.010809

Top 2 features:
            feature  std_coef  abs_influence
8   color_intensity  0.394276       0.394276
11          proline  0.298278       0.298278


In [ ]:
#Classification Showdown
#Goal: Predict the wine cultivar ("Target": 0,1,2)

# import numpy as np
# import pandas as pd
# from sklearn.datasets import load_wine
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
# from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# wine = load_wine()

# print(wine.feature_names)
# df = pd.DataFrame(wine.data, columns=wine.feature_names)
# df["cultivar"] = wine.target
# y = df["cultivar"]
# X = df.drop(columns=["cultivar", "target"], errors="ignore")

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# from sklearn.linear_model import LogisticRegression

# model = LogisticRegression(max_iter=10000)
# model.fit(X_train, y_train)

# print(df.columns.tolist())

import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load features (13 chemical measurements) and target (cultivars 0, 1, 2)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

# 2. Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Build pipeline: scale features first, then fit multiclass LogisticRegression
clf_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

# 4. Cross-validation on full dataset
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf_pipe, X, y, cv=cv, scoring="accuracy")
print(f"5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# 5. Fit on training set and evaluate on held-out test set
clf_pipe.fit(X_train, y_train)
y_pred = clf_pipe.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Test Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(
    "\nClassification Report:\n",
    classification_report(y_test, y_pred, target_names=wine.target_names),
)

# 6. Inspect per-class standardized coefficients
model = clf_pipe.named_steps["logisticregression"]
coef_df = pd.DataFrame(model.coef_, columns=X.columns, index=wine.target_names).T
coef_df["max_abs_coef"] = coef_df.abs().max(axis=1)
print("\nTop 5 features by max |coef| across cultivars:")
print(coef_df.sort_values("max_abs_coef", ascending=False).head(5).round(4))

5-Fold CV Accuracy: 0.9833 (+/- 0.0136)
Test Accuracy: 0.9722
Test Macro F1: 0.9710

Confusion Matrix:
 [[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]

Classification Report:
               precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36


Top 5 features by max |coef| across cultivars:
                   class_0  class_1  class_2  max_abs_coef
color_intensity     0.2202  -1.0960   0.8758        1.0960
proline             0.9265  -1.0099   0.0834        1.0099
flavanoids          0.7053   0.1982  -0.9035        0.9035
alcohol             0.7175  -0.8615   0.1441        0.8615
alcalinity_of_ash  -0.8351   0.4662   0.3688        0.8351


In [1]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 1. Load dataset (13 chemical features -> target cultivars 0, 1, 2)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Define models in progression order
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "KNN (k=5)": make_pipeline(
        StandardScaler(), KNeighborsClassifier(n_neighbors=5)
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

# 3. Evaluate each model with CV and on the held-out test set
for name, clf in models.items():
  cv_scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)

  print(f"=== {name} ===")
  print(f"5-Fold CV Acc : {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
  print(f"Test Acc      : {accuracy_score(y_test, y_pred):.4f}")
  print(f"Test Macro F1 : {f1_score(y_test, y_pred, average='macro'):.4f}")
  print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
  print(
      classification_report(
          y_test, y_pred, target_names=wine.target_names, digits=4
      )
  )

# 4. Inspect top feature importances from Decision Tree and Random Forest
dt_model = models["Decision Tree"]
rf_model = models["Random Forest"]

print("Top 3 Decision Tree features:")
print(
    pd.Series(dt_model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(3)
    .round(4)
)

print("\nTop 3 Random Forest features:")
print(
    pd.Series(rf_model.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
    .head(3)
    .round(4)
)

=== Logistic Regression ===
5-Fold CV Acc : 0.9833 (+/- 0.0136)
Test Acc      : 0.9722
Test Macro F1 : 0.9710
Confusion Matrix:
 [[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]
              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        12
     class_1     0.9333    1.0000    0.9655        14
     class_2     1.0000    0.9000    0.9474        10

    accuracy                         0.9722        36
   macro avg     0.9778    0.9667    0.9710        36
weighted avg     0.9741    0.9722    0.9720        36

=== KNN (k=5) ===
5-Fold CV Acc : 0.9717 (+/- 0.0181)
Test Acc      : 0.9722
Test Macro F1 : 0.9718
Confusion Matrix:
 [[12  0  0]
 [ 0 13  1]
 [ 0  0 10]]
              precision    recall  f1-score   support

     class_0     1.0000    1.0000    1.0000        12
     class_1     1.0000    0.9286    0.9630        14
     class_2     0.9091    1.0000    0.9524        10

    accuracy                         0.9722        36
   macro avg     0.9697  

In [2]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. KNN on raw/unscaled features (before StandardScaler)
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_cv = cross_val_score(knn_unscaled, X, y, cv=cv, scoring="accuracy")
knn_unscaled.fit(X_train, y_train)
knn_pred = knn_unscaled.predict(X_test)

print(f"KNN (unscaled) -> 5-Fold CV Acc: {knn_cv.mean():.4f} | Test Acc: {accuracy_score(y_test, knn_pred):.4f} | Macro F1: {f1_score(y_test, knn_pred, average='macro'):.4f}")
print("KNN Confusion Matrix:\n", confusion_matrix(y_test, knn_pred))

# 2. Logistic Regression after StandardScaler
logreg_scaled = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
lr_cv = cross_val_score(logreg_scaled, X, y, cv=cv, scoring="accuracy")
logreg_scaled.fit(X_train, y_train)
lr_pred = logreg_scaled.predict(X_test)

print(f"\nLogistic Regression (scaled) -> 5-Fold CV Acc: {lr_cv.mean():.4f} | Test Acc: {accuracy_score(y_test, lr_pred):.4f} | Macro F1: {f1_score(y_test, lr_pred, average='macro'):.4f}")
print("LogReg Confusion Matrix:\n", confusion_matrix(y_test, lr_pred))

KNN (unscaled) -> 5-Fold CV Acc: 0.6802 | Test Acc: 0.8056 | Macro F1: 0.8025
KNN Confusion Matrix:
 [[12  0  0]
 [ 0 10  4]
 [ 0  3  7]]

Logistic Regression (scaled) -> 5-Fold CV Acc: 0.9833 | Test Acc: 0.9722 | Macro F1: 0.9710
LogReg Confusion Matrix:
 [[12  0  0]
 [ 0 14  0]
 [ 0  1  9]]


In [4]:
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 1. Track KNN accuracy jump: unscaled vs. after StandardScaler
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

cv_before = cross_val_score(knn_unscaled, X, y, cv=cv, scoring="accuracy").mean()
cv_after = cross_val_score(knn_scaled, X, y, cv=cv, scoring="accuracy").mean()

knn_unscaled.fit(X_train, y_train)
knn_scaled.fit(X_train, y_train)

test_before = accuracy_score(y_test, knn_unscaled.predict(X_test))
test_after = accuracy_score(y_test, knn_scaled.predict(X_test))

cv_jump = cv_after - cv_before
test_jump = test_after - test_before

print(
    f"KNN CV Accuracy Jump  : {cv_before:.4f} -> {cv_after:.4f} ({cv_jump:+.4f} / {cv_jump*100:+.2f} pp)"
)
print(
    f"KNN Test Accuracy Jump: {test_before:.4f} -> {test_after:.4f} ({test_jump:+.4f} / {test_jump*100:+.2f} pp)"
)

# 2. Compare all 4 models in a quick summary table
models = {
    "KNN (k=5, scaled)": knn_scaled,
    "Logistic Regression (scaled)": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, clf in models.items():
  cv_acc = cross_val_score(clf, X, y, cv=cv, scoring="accuracy").mean()
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)
  rows.append({
      "Model": name,
      "5-Fold CV Acc": round(cv_acc, 4),
      "Test Acc": round(accuracy_score(y_test, y_pred), 4),
      "Test Macro F1": round(f1_score(y_test, y_pred, average="macro"), 4),
  })

summary_table = pd.DataFrame(rows)
print("\n4-Model Summary Table:")
print(summary_table.to_string(index=False))

KNN CV Accuracy Jump  : 0.6802 -> 0.9717 (+0.2916 / +29.16 pp)
KNN Test Accuracy Jump: 0.8056 -> 0.9722 (+0.1667 / +16.67 pp)

4-Model Summary Table:
                       Model  5-Fold CV Acc  Test Acc  Test Macro F1
           KNN (k=5, scaled)         0.9717    0.9722         0.9718
Logistic Regression (scaled)         0.9833    0.9722         0.9710
               Decision Tree         0.8932    0.9444         0.9457
               Random Forest         0.9775    1.0000         1.0000


In [5]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# 1. Load features only (pretend target is not available during clustering)
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y_true = pd.Series(wine.target, name="true_class")

# 2. Scale features so distance metrics treat all 13 chemical features evenly
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Run K-Means with 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
clusters = pd.Series(kmeans.fit_predict(X_scaled), name="cluster")

# 4. Compare unsupervised cluster IDs against actual wine classes
ct = pd.crosstab(clusters, y_true)
print("Crosstab (KMeans cluster vs true_class):\n")
print(ct)

# Optional: check majority-matched alignment and Adjusted Rand Index (ARI)
matched_samples = ct.max(axis=1).sum()
print(f"\nMajority-matched samples: {matched_samples}/{len(y_true)} ({matched_samples / len(y_true):.4f})")
print(f"Adjusted Rand Index (ARI): {adjusted_rand_score(y_true, clusters):.4f}")

Crosstab (KMeans cluster vs true_class):

true_class   0   1   2
cluster               
0            0  65   0
1            0   3  48
2           59   3   0

Majority-matched samples: 172/178 (0.9663)
Adjusted Rand Index (ARI): 0.8975
